In [1]:
from math import pi
import numpy as np
import scipy as sp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
from qiskit.primitives import Estimator # new
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
import os
#from sko.SA import SA

In [2]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    # Function to evaluate the expectation value of the Hamiltonian for a given set of parameters
    # Inputs: parameter_values (ndarray), parameter values
    # Outputs: result (float), energy value

    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars =  list(value_dict.values())
    expectation_value = estimator.run(ansatz,qubit_op,pars).result().values
    return np.real(expectation_value)

In [7]:
import numpy as np
import time
from typing import Callable
from qiskit_algorithms.optimizers import OptimizerResult, OptimizerSupportLevel

class HyDEOptimizer:
    def __init__(self, I_NP=3000, F_weight=0.5, F_CR=0.6, I_itermax=3000, 
                 I_strategy=1, I_strategyVersion=1, I_bnd_constr=1, 
                 VarMin=-500, VarMax=500, qubits= 1):
        self.I_NP = I_NP
        self.F_weight = F_weight
        self.F_CR = F_CR
        self.I_itermax = I_itermax
        self.I_strategy = I_strategy
        self.I_strategyVersion = I_strategyVersion
        self.I_bnd_constr = I_bnd_constr
        self.VarMin = VarMin
        self.VarMax = VarMax
        self.qubits = qubits

    def get_support_level(self):
        return {
            "gradient": OptimizerSupportLevel.ignored,
            "bounds": OptimizerSupportLevel.required,
            "initial_point": OptimizerSupportLevel.required,
        }

    def minimize(self, fun: Callable, x0: np.ndarray) -> OptimizerResult:
        return self._hyde(fun, x0)

    def _hyde(self, fun, x0):
        start_time = time.time()
        de_parameters = {
            'I_NP': self.I_NP,
            'F_weight': self.F_weight,
            'F_CR': self.F_CR,
            'I_itermax': self.I_itermax,
            'I_strategy': self.I_strategy,
            'I_strategyVersion': self.I_strategyVersion,
            'I_bnd_constr': self.I_bnd_constr,
            'nVariables': len(x0),
            'minPositionsMatrix': np.tile(self.VarMin, (self.I_NP, len(x0))),
            'maxPositionsMatrix': np.tile(self.VarMax, (self.I_NP, len(x0)))
        }

        other_parameters = {
            'iRuns': np.random.randint(1000000),  # For reproducibility in the example, it can be adjusted
            'fnc': fun
        }

        low_habitat_limit = self.VarMin * np.ones(len(x0))
        up_habitat_limit = self.VarMax * np.ones(len(x0))

        # Initialization
        np.random.seed(other_parameters['iRuns'])
        FM_pop = self.genpop(de_parameters['I_NP'], len(x0), de_parameters['minPositionsMatrix'], de_parameters['maxPositionsMatrix'])

        S_val = np.array([fun(ind) for ind in FM_pop])
        I_best_index = np.argmin(S_val)
        FVr_bestmemit = FM_pop[I_best_index, :]
        fit_max_vector = np.zeros(de_parameters['I_itermax'])
        fit_max_vector[0] = S_val[I_best_index]

        FVr_rot = np.arange(de_parameters['I_NP'])
        F_weight_old, F_CR_old = None, None
        if de_parameters['I_strategy'] == 3:
            F_weight_old = np.tile(de_parameters['F_weight'], (de_parameters['I_NP'], 3))
            de_parameters['F_weight'] = F_weight_old
            F_CR_old = np.tile(de_parameters['F_CR'], (de_parameters['I_NP'], 1))
            de_parameters['F_CR'] = F_CR_old

        I_strategy_version = de_parameters['I_strategyVersion']
        other = {}

        gen = 1
        while np.abs(fit_max_vector[gen-1] + self.qubits - 1) >= 1e-1:
            other['a'] = (de_parameters['I_itermax'] - gen) / de_parameters['I_itermax']
            other['lowerlimit'] = low_habitat_limit
            other['upperlimit'] = up_habitat_limit

            if de_parameters['I_strategy'] == 3:
                value_R = np.random.rand(de_parameters['I_NP'], 3)
                ind1 = value_R < 0.1
                ind2 = np.random.rand(de_parameters['I_NP'], 1) < 0.1
                de_parameters['F_weight'][ind1] = 0.1 + np.random.rand(np.sum(ind1)) * 0.9
                de_parameters['F_weight'][~ind1] = F_weight_old[~ind1]
                de_parameters['F_CR'][ind2] = np.random.rand(np.sum(ind2))
                de_parameters['F_CR'][~ind2] = F_CR_old[~ind2]

            FM_ui, FM_base, _ = self.generate_trial(de_parameters['I_strategy'], de_parameters['F_weight'], de_parameters['F_CR'], FM_pop, FVr_bestmemit, de_parameters['I_NP'], len(x0), FVr_rot, I_strategy_version, other)
            FM_ui = self.update(FM_ui, de_parameters['minPositionsMatrix'], de_parameters['maxPositionsMatrix'], de_parameters['I_bnd_constr'], FM_base)
            S_val_temp = np.array([fun(ind) for ind in FM_ui])

            ind = np.where(S_val_temp < S_val)[0]
            S_val[ind] = S_val_temp[ind]
            FM_pop[ind, :] = FM_ui[ind, :]

            S_bestval = np.min(S_val)
            I_best_index = np.argmin(S_val)
            FVr_bestmemit = FM_pop[I_best_index, :]
            fit_max_vector[gen] = S_bestval

            if de_parameters['I_strategy'] == 3:
                F_weight_old[ind, :] = de_parameters['F_weight'][ind, :]
                F_CR_old[ind] = de_parameters['F_CR'][ind]
            if gen % 50 == 1:
                print(f'Fitness value: {fit_max_vector[gen]}')
                print(f'Generation: {gen}')
            gen += 1

        result = OptimizerResult()
        result.x = FVr_bestmemit
        result.fun = fit_max_vector[gen-1]
        result.nfev = gen * de_parameters['I_NP']
        result.njev = None
        result.nit = gen
        result.time = time.time() - start_time

        return result

    def genpop(self, a, b, low_matrix, up_matrix):
        return np.random.uniform(low=low_matrix, high=up_matrix, size=(a, b))

    def update(self, p, low_matrix, up_matrix, BRM, FM_base):
        if BRM == 1:
            p = np.clip(p, low_matrix, up_matrix)
        elif BRM == 2:
            idx = np.where((p < low_matrix) | (p > up_matrix))
            p[idx] = np.random.uniform(low=low_matrix[idx], high=up_matrix[idx])
        elif BRM == 3:
            idx = np.where(p < low_matrix)
            p[idx] = np.random.uniform(low=low_matrix[idx], high=FM_base[idx])
            idx = np.where(p > up_matrix)
            p[idx] = np.random.uniform(low=FM_base[idx], high=up_matrix[idx])
        return p

    def generate_trial(self, method, F_weight, F_CR, FM_pop, FVr_bestmemit, I_NP, I_D, FVr_rot, I_strategy_version, other):
        FM_popold = FM_pop
        FVr_ind = np.random.permutation(4)
        FVr_a1 = np.random.permutation(I_NP)
        FVr_rt = (FVr_rot + FVr_ind[0]) % I_NP
        FVr_a2 = FVr_a1[FVr_rt]
        FVr_rt = (FVr_rot + FVr_ind[1]) % I_NP
        FVr_a3 = FVr_a2[FVr_rt]

        FM_pm1 = FM_popold[FVr_a1, :]
        FM_pm2 = FM_popold[FVr_a2, :]
        FM_pm3 = FM_popold[FVr_a3, :]

        FM_mui = np.random.rand(I_NP, I_D) < F_CR
        FM_mpo = FM_mui < 0.5
        if method == 1: #DE/rand1
            FM_ui = FM_pm3 + F_weight * (FM_pm1 - FM_pm2)
            FM_ui = FM_popold * FM_mpo + FM_ui * FM_mui
            FM_base = FM_pm3
        elif method == 2: # DE/current-to-best/1
            FM_bm = np.tile(FVr_bestmemit, (I_NP, 1))
            FM_ui = FM_popold + F_weight * (FM_bm - FM_popold) + F_weight * (FM_pm1 - FM_pm2)
            FM_ui = FM_popold * FM_mpo + FM_ui * FM_mui
            FM_base = FM_bm
        elif method == 3:
            FM_bm = np.tile(FVr_bestmemit, (I_NP, 1))
            if len(F_weight) == 1:
                FM_ui = FM_popold + F_weight * (FM_bm - FM_popold) + F_weight * (FM_pm1 - FM_pm2)
            else:
                if I_strategy_version == 1: #HyDE-DF
                    a = other['a']
                    ginv = np.log((1 + a) / a)
                    FM_ui = FM_popold + np.tile(F_weight[:, 1].reshape(-1, 1), (1, I_D)) * (FM_bm - FM_popold) + np.tile(F_weight[:, 0].reshape(-1, 1), (1, I_D)) * (FM_pm1 - FM_pm2) + np.tile(F_weight[:, 1].reshape(-1, 1), (1, I_D)) * ginv * (np.random.rand(I_NP, I_D) - 0.5)
                elif I_strategy_version == 2: # HyDE
                    FM_ui = FM_popold + np.tile(F_weight[:, 1].reshape(-1, 1), (1, I_D)) * (FM_bm - FM_popold) + np.tile(F_weight[:, 0].reshape(-1, 1), (1, I_D)) * (FM_pm1 - FM_pm2)
            FM_ui = FM_popold * FM_mpo + FM_ui * FM_mui
            FM_base = FM_bm
        return FM_ui, FM_base, FM_mpo


In [ ]:
min_qubits = 3
max_qubits=10
for k in range(min_qubits,max_qubits):
    #################################################### Hamiltonian ###########################################################

    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension

    ###################################################### Ansatz ##############################################################

    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))

    print('ansatz_num_parameters=',ansatz.num_parameters)

    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map

    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )

    optimizer = HyDEOptimizer(I_NP=50,I_itermax=10000 , I_strategy=2, I_strategyVersion=1, qubits=k)
    x0 = np.random.uniform(-5.12, 5.12, dimension)
    result = optimizer.minimize(evaluate_expectation, x0)

    print(result)

ansatz_num_parameters= 12


C:\Users\petre\AppData\Local\Temp\ipykernel_47456\3756344769.py:62: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fit_max_vector[0] = S_val[I_best_index]


Fitness value: -1.46875
Generation: 1
Fitness value: -1.84375
Generation: 51
Fitness value: -1.84375
Generation: 101
{   'fun': -1.96875,
    'jac': None,
    'nfev': 6350,
    'nit': 127,
    'njev': None,
    'time': 17.417169332504272,
    'x': array([ 461.50092482, -321.28250951,   -0.57147902, -151.94849634,
       -246.41960969,   -4.82958487,  117.69359249,  127.05086635,
        165.21802092, -388.35667014,  158.78908356, -231.38279877])}
ansatz_num_parameters= 16
Fitness value: -1.25
Generation: 1
Fitness value: -2.03125
Generation: 51
Fitness value: -2.34375
Generation: 101
Fitness value: -2.34375
Generation: 151
Fitness value: -2.34375
Generation: 201
Fitness value: -2.5
Generation: 251
Fitness value: -2.625
Generation: 301
Fitness value: -2.625
Generation: 351
Fitness value: -2.8125
Generation: 401
Fitness value: -2.8125
Generation: 451
Fitness value: -2.8125
Generation: 501
Fitness value: -2.8125
Generation: 551
Fitness value: -2.8125
Generation: 601
Fitness value: -2.8125

KeyboardInterrupt: 